# subsetting surface currents

In [1]:
import time
import icechunk
import xarray as xr
import xarray_subset_grid as xsg


In [2]:
ENDPOINT = "https://usgs.osn.mghpcc.org"
BUCKET = "esip"

stores = ["eccofs-avg-temp-balanced-10mb", "eccofs-avg-temp-map-10mb",
          "eccofs-avg-uv-balanced-10mb", "eccofs-avg-uv-map-10mb"]

def open_eccofs(name):
    """Open s3://esip/rsignell/<name> anonymously as an xarray Dataset (lazy, dask-backed)."""
    storage = icechunk.s3_storage(
        bucket=BUCKET, prefix=f"rsignell/{name}", region="us-east-1",
        endpoint_url=ENDPOINT, anonymous=True, force_path_style=True,
    )
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")
    return xr.open_zarr(session.store, consolidated=False)

In [3]:
# ds = open_eccofs("eccofs-avg-uv-map-10mb") # took 1.59 minutes
ds = open_eccofs("eccofs-avg-uv-balanced-10mb") # took .39 minutes


In [4]:
ds


<xarray.Dataset> Size: 161GB
Dimensions:      (eta_rho: 1443, xi_rho: 1667, s_w: 51, eta_psi: 1442,
                  xi_psi: 1666, s_rho: 50, eta_v: 1442, xi_v: 1667,
                  eta_u: 1443, xi_u: 1666, time: 166)
Coordinates:
  * s_w          (s_w) float64 408B -1.0 -0.98 -0.96 -0.94 ... -0.04 -0.02 0.0
  * s_rho        (s_rho) float64 400B -0.99 -0.97 -0.95 ... -0.05 -0.03 -0.01
    lat_v        (eta_v, xi_v) float32 10MB dask.array<chunksize=(144, 167), meta=np.ndarray>
    lon_v        (eta_v, xi_v) float32 10MB dask.array<chunksize=(144, 167), meta=np.ndarray>
    lat_u        (eta_u, xi_u) float32 10MB dask.array<chunksize=(144, 167), meta=np.ndarray>
    lon_u        (eta_u, xi_u) float32 10MB dask.array<chunksize=(144, 167), meta=np.ndarray>
  * time         (time) datetime64[ns] 1kB 2026-06-09T12:00:00 ... 2026-11-21...
Dimensions without coordinates: eta_rho, xi_rho, eta_psi, xi_psi, eta_v, xi_v,
                                eta_u, xi_u
Data variables: (12/25)
    angle        (eta_rho, xi_rho) float32 10MB dask.array<chunksize=(481, 556), meta=np.ndarray>
    Tcline       float64 8B ...
    Vtransform   int32 4B ...
    f            (eta_rho, xi_rho) float32 10MB dask.array<chunksize=(481, 556), meta=np.ndarray>
    h            (eta_rho, xi_rho) float32 10MB dask.array<chunksize=(481, 556), meta=np.ndarray>
    hc           float64 8B ...
    ...           ...
    theta_b      float64 8B ...
    theta_s      float64 8B ...
    u            (time, s_rho, eta_u, xi_u) float32 80GB dask.array<chunksize=(17, 5, 144, 167), meta=np.ndarray>
    zeta         (time, eta_rho, xi_rho) float32 2GB dask.array<chunksize=(28, 240, 278), meta=np.ndarray>
    pm           (eta_rho, xi_rho) float32 10MB dask.array<chunksize=(481, 556), meta=np.ndarray>
    v            (time, s_rho, eta_v, xi_v) float32 80GB dask.array<chunksize=(17, 5, 144, 167), meta=np.ndarray>
Attributes: (12/44)
    format:            netCDF-3 64bit offset file
    Conventions:       CF-1.4, SGRID-0.3
    type:              ROMS nonlinear model averages file
    title:             ECCOFS forward run
    var_info:          varinfo.yaml
    rst_file:          eccofs_rst_20261115.nc
    ...                ...
    compiler_flags:    -frepack-arrays -fallow-argument-mismatch        -O3 -...
    tiling:            32x12
    ana_file:          ROMS/Functionals/ana_btflux.h
    CPP_options:       ECCOFS, ADD_FSOBC, ADD_M2OBC, ANA_BSFLUX, ANA_BTFLUX, ...
    sources:           ['20260608_0002', '20260609_0002', '20260610_0002', '2...
    latest_run:        20261115

In [5]:
[*ds]


['angle',
 'Tcline',
 'Vtransform',
 'f',
 'h',
 'hc',
 'grid',
 'Cs_w',
 'lat_psi',
 'lon_psi',
 'lon_rho',
 'lat_rho',
 'Cs_r',
 'mask_psi',
 'mask_rho',
 'mask_v',
 'pn',
 'Vstretching',
 'mask_u',
 'theta_b',
 'theta_s',
 'u',
 'zeta',
 'pm',
 'v']

In [6]:
# region around gulf Florida Strait
# lat: 23 -- 25.5
# lon: -84.0 -- -78.0
BBox = (-84.0, 23.0, -78, 25.5)

vars_to_keep = ['Cs_r',
 'angle',
 'h',
 'grid',
 'mask_v',
 'mask_rho',
 'u',
 'mask_u',
 'mask_psi',
 'zeta',
 'v']

In [7]:
ds_vars = ds.xsg.subset_vars(vars_to_keep)


In [8]:
ds_surf = ds_vars.xsg.subset_vertical_level(level=0, method='nearest')

In [9]:
print(ds_vars.nbytes, ds_vars.nbytes / ds_surf.nbytes)

161352560224 32.766686390511204


In [10]:
ds_subset = ds_surf.xsg.subset_bbox(BBox)

/tmp/ipykernel_2905/4089246912.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_subset = ds_surf.xsg.subset_bbox(BBox)


In [11]:
print(ds_subset.nbytes, ds_surf.nbytes / ds_subset.nbytes)

87463080 56.30131900225787


In [12]:
start = time.time()
ds_subset.to_netcdf('ECOFS_subset.nc')
print(f"It took {(time.time() - start) / 60:.2f} Minutes to write the netCDF")

It took 0.39 Minutes to write the netCDF
